In [2]:
!pip install -q langchain langchain-chroma langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 89.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.1 MB/

In [3]:
import os
import chromadb
from chromadb.errors import InvalidCollectionException
from tqdm import tqdm
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
client = chromadb.Client()

In [5]:
DATA_DIR = "/kaggle/input/bophapdien/vbpl"

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="bkai-foundation-models/vietnamese-bi-encoder", model_kwargs={"device": "cuda"})

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=20 
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

In [7]:
# Helper for data ingestion

def get_data(data_dir):
    """Chunk the data and add metadata"""
    documents = []
    files = os.listdir(data_dir)

    for file in tqdm(files):
        path = os.path.join(data_dir, file)
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Chunk the text
        chunks = text_splitter.split_text(content)

        for chunk in chunks:
            document = Document(
                page_content=chunk,
                metadata={"file_path": path}
            )
            
            documents.append(document)
    
    return documents


def get_vectorstore(client, embedding_model, collection_name, text_dir, persist_dir=None):
    """Load the vectorstore. If not created, embed and add the documents to the collection

    Args:
        client: Chroma ClientAPI
        embedding_model: Any embedding model
        collection_name: name of the collection
        text_dir: directory of html files
        persist_dir: persistant folder

    """
    try:
        client.get_collection(collection_name)

        # If collection exist (already ingested), simply load the chroma
        vectorstore = Chroma(
            client=client,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir
        )

    except InvalidCollectionException: # collection does not exist
        # Create new collection and ingest the data
        doc_list = get_data(text_dir)

        vectorstore = Chroma.from_documents(
            client=client,
            documents=doc_list,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir
        )

    return vectorstore

In [8]:
vector_store = get_vectorstore(client=client, embedding_model=embedding_model, collection_name="isods", text_dir=DATA_DIR)

100%|██████████| 5943/5943 [00:49<00:00, 121.10it/s]


In [9]:
# Semantic search vector store

def semantic_search(query, k):
    results = vector_store.similarity_search_by_vector(embedding=embedding_model.embed_query(query),k=k, )
    for doc in results:
        print(f"* {doc.page_content} \n [{doc.metadata}] \n ------")
    return

In [10]:
k = 3
semantic_search(query="Quy định về sử dụng mũ bảo hiểm khi lái xe", k=k)

* <p>
	c) Trường hợp mũ bảo hiểm có lưỡi trai cứng gắn liền với vỏ mũ thì độ dài của lưỡi trai cứng tính từ điểm kết nối với vỏ mũ đến điểm xa nhất của lưỡi trai không được lớn hơn 50 mm và góc nghiêng của lưỡi trai không được làm ảnh hưởng đến góc nhìn theo quy định tại quy chuẩn kỹ thuật quốc gia QCVN 2 &#58; 2008/BKHCN;</p>
<p>
	d) Trường hợp mũ bảo hiểm có vành cứng xung quanh thì không được nhô quá 20 mm.</p>
<p align="center">
	<strong>Chương </strong><strong><a name="Chuong_II"></a>II</strong></p>
<p align="center">
	<strong>QUY ĐỊNH CỤ THỂ</strong></p>
<p>
	<strong>Điều <a name="Chuong_II_Dieu_5"></a>5. Trách nhiệm của tổ chức, cá nhân sản xuất mũ bảo hiểm</strong></p>
<p>
	1. Thực hiện việc chứng nhận hợp quy, công bố hợp quy đối với mũ bảo hiểm do mình sản xuất theo quy chuẩn kỹ thuật quốc gia QCVN 2 &#58; 2008/BKHCN, gắn dấu hợp quy CR và ghi nhãn hàng hóa theo quy định của pháp luật về nhãn hàng hóa trước khi đưa ra lưu thông trên thị trường.</p>
<p>
	2. Chịu trách nhiệm về

In [12]:
k = 3
semantic_search(query="Vượt đèn đỏ bị phạt bao nhiêu tiền", k=k)

* <p>
	4. Phạt tiền từ 200.000 đồng đến 300.000 đồng đối với người điều khiển xe mô tô, xe gắn máy (kể cả xe máy điện), các loại xe tương tự xe mô tô và các loại xe tương tự xe gắn máy dừng xe, đỗ xe trong phạm vi an toàn đường ngang, cầu chung; không chấp hành hiệu lệnh, chỉ dẫn của biển báo hiệu, vạch kẻ đường khi đi qua đường ngang, cầu chung.</p>
<p>
	5. Phạt tiền từ 600.000 đồng đến 1.000.000 đồng đối với người điều khiển xe mô tô, xe gắn máy (kể cả xe máy điện), các loại xe tương tự xe mô tô và các loại xe tương tự xe gắn máy vượt rào chắn đường ngang, cầu chung khi chắn đang dịch chuyển; vượt đường ngang, cầu chung khi đèn đỏ đã bật sáng; không chấp hành hiệu lệnh, chỉ dẫn của nhân viên gác đường ngang, cầu chung khi đi qua đường ngang, cầu chung.</p>
<p>
	6. Phạt tiền từ 800.000 đồng đến 1.000.000 đồng đối với người điều khiển xe ô tô, các loại xe tương tự xe ô tô, máy kéo, xe máy chuyên dùng dừng xe, đỗ xe quay đầu xe trong phạm vi an toàn đường ngang, cầu chung; không chấp hà

In [15]:
k = 1
semantic_search(query="Ăn cắp tài sản bị chế tài như nào", k=k)

* <p>
	1. Người nào bằng thủ đoạn gian dối chiếm đoạt tài sản của người khác trị giá từ 2.000.000 đồng đến dưới 50.000.000 đồng hoặc dưới 2.000.000 đồng nhưng thuộc một trong các trường hợp sau đây, thì bị phạt cải tạo không giam giữ đến 03 năm hoặc phạt tù từ 06 tháng đến 03 năm&#58;</p>
<p>
	a) Đã bị xử phạt vi phạm hành chính về hành vi chiếm đoạt tài sản mà còn vi phạm;</p>
<p>
	b) Đã bị kết án về tội này hoặc về một trong các tội quy định tại các điều 168, 169, 170, 171, 172, 173, 175 và 290 của Bộ luật này, chưa được xóa án tích mà còn vi phạm;</p>
<p>
	c) Gây ảnh hưởng xấu đến an ninh, trật tự, an toàn xã hội;</p>
<p>
	d) Tài sản là phương tiện kiếm sống chính của người bị hại và gia đình họ; tài sản là kỷ vật, di vật, đồ thờ cúng có giá trị đặc biệt về mặt tinh thần đối với người bị hại.</p>
<p>
	2. Phạm tội thuộc một trong các trường hợp sau đây, thì bị phạt tù từ 02 năm đến 07 năm&#58;</p>
<p>
	a) Có tổ chức;</p>
<p>
	b) Có tính chất chuyên nghiệp;</p>
<p>
	c) Chiếm đoạt tài 

In [16]:
k = 5
semantic_search(query="Chiếm đoạt tài sản trị giá bao nhiêu thì bị đi tù chung thân", k=k)

* <p>
	c) Chiếm đoạt tài sản trị giá từ 50.000.000 đồng đến dưới 200.000.000 đồng; d) Lợi dụng chức vụ, quyền hạn hoặc lợi dụng danh nghĩa cơ quan, tổ chức; đ) Dùng thủ đoạn xảo quyệt;</p>
<p>
	e) Tái phạm nguy hiểm.</p>
<p>
	3. Phạm tội thuộc một trong các trường hợp sau đây, thì bị phạt tù từ 05 năm đến 12 năm&#58;</p>
<p>
	a) Chiếm đoạt tài sản trị giá từ 200.000.000 đồng đến dưới 500.000.000 đồng;</p>
<p>
	b) Gây ảnh hưởng xấu đến an ninh, trật tự, an toàn xã hội.</p>
<p>
	4. Phạm tội chiếm đoạt tài sản trị giá 500.000.000 đồng trở lên, thì bị phạt tù từ 12 năm đến 20 năm.</p>
<p>
	5. Người phạm tội còn có thể bị phạt tiền từ 10.000.000 đồng đến 100.000.000 đồng, bị cấm đảm nhiệm chức vụ, cấm hành nghề hoặc làm công việc nhất định từ 01 năm đến 05 năm hoặc tịch thu một phần hoặc toàn bộ tài sản.</p>
<p>
	<strong>Điều <a name="Phan_hai_Chuong_XVI_Dieu_176"></a>176. Tội chiếm giữ trái phép tài sản</strong></p>
<p>
	1. Người nào cố tình không trả lại cho chủ sở hữu, người quản lý hợp 

In [18]:
k = 2
semantic_search(query="Trộm xe đạp bị cảnh sát bắt thì làm sao", k=k)

* <p>
	d) Trường hợp chỉ áp dụng hình thức phạt tiền thì cán bộ Cảnh sát giao thông tạm giữ một trong các loại giấy tờ theo thứ tự (trừ khi các giấy tờ đó có dấu hiệu nghi giả, cần xác minh để làm rõ hành vi vi phạm thì được giữ thêm giấy tờ khác có liên quan)&#58; Giấy phép lái xe, Chứng chỉ bồi dưỡng kiến thức pháp luật về giao thông đường bộ hoặc Giấy đăng ký xe hoặc bản sao chứng thực Giấy đăng ký xe kèm bản gốc Giấy biên nhận của tổ chức tín dụng còn hiệu lực (trong thời gian tổ chức tín dụng giữ bản chính Giấy đăng ký xe) hoặc Giấy chứng nhận kiểm định an toàn kỹ thuật và bảo vệ môi trường, Giấy xác nhận thời hạn hiệu lực của Giấy chứng nhận kiểm định và Tem kiểm định (đối với loại phương tiện giao thông có quy định phải kiểm định) hoặc giấy tờ cần thiết khác có liên quan đến tang vật, phương tiện theo quy định của pháp luật để bảo đảm cho việc thi hành quyết định xử phạt;</p>
<p>
	đ) Trường hợp giao phương tiện giao thông bị tạm giữ để bảo đảm thi hành quyết định xử phạt cho ngư

## Some comment on the retrieval

- Despite ingesting the whole HTML document, the vector store still managed to retrieve parts with relevant text only.
- When asked about information at the end of chunks (such as "Chiếm đoạt tài sản bao nhiêu thì bị phạt tù chung thân" that is in the previous question's chunk), the model might not retrieve the correct result. This might be due to the max sequence length of the embedding model (only 256).
- When asked using synonyms ,such as "trộm" ~ "ăn cắp", "cảnh sát" ~ "công an", the vector store will struggle to return the best ranking.